## Hybrid search strategies

# 1. Dense and sparse retrieval

Sparse-Retrieval:  {EXACT SEARCH}  Matches the exact words using techiniques such as TF-IDF(Term Frequency-Inverse Document Frequency) which allows us exact search for the query. There are several algorithms for NLP such as TF-IDF,BM-25 etc which helps in generating the sparse matrix (The matrix of 0's and 1's)

Dense-Retrieval: {SEMANTIC MEANING} the output generated as a result of semantic matching by vector embeddings. Here , we get a Dense matrix rather than a sparse matrix. (Normal way which we already done)

HYBRID SEARCH : Sparse Retrieval + Dense Retrievel


score(hyb)= @ x score(dense)+ (1-@)*score(parse)

score(dense)=cosine similarity(Input,vectorstore)

score(parse)=TF-IDF(Input,vectorStoreRepresentation)

@=weightage given with respect to sparse and dense.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
loaded_text = TextLoader("DandS.txt", encoding="utf-8")
docs = loaded_text.load()

# Split into smaller semantic units so hybrid retrieval can return multiple relevant hits.
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_documents(docs)
print(f"Loaded {len(docs)} base doc(s), split into {len(chunks)} chunk(s)")

Loaded 1 base doc(s), split into 4 chunk(s)


In [ ]:
import os

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment before running embeddings.")

In [ ]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
from langchain_community.vectorstores import FAISS

dense_vector_store = FAISS.from_documents(chunks, embeddings)
dense_retriever = dense_vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
## SPARSE RETRIEVER USING BM-25 ( An algorithm which converts text into vectors )
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 3  # top k documents to retrieve

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],
)

In [ ]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B7EC922AD0>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000002B7EC922C10>, k=3)], weights=[0.7, 0.3])

In [ ]:
query = "How can I build applications using LLMs?"
results = hybrid_retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\nDocument {i}:\n{doc.page_content}")


Document 1:
Langchain is a framework for buliding applications using LLMs.

Document 2:
You can create chains,memory , agents and retrievers.
The Eiffel tower is located in Paris.

Document 3:
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Document 4:
France is a Popular tourist destination.
